# Ascent-DTwin — Jupyter Demo (DTaaS-style workspace)
Query live twin telemetry from the Ascent API + InfluxDB, plot it, and push synthetic data.
Inspired by INTO-CPS DTaaS user workspaces (JupyterLab).

In [ ]:
import json, urllib.request
# Inside the docker network the API is at ascent-api:8000.
# If you run this notebook from your host browser, localhost:8000 also works.
API = "http://ascent-api:8000"
try:
    print(urllib.request.urlopen(API+"/api/health", timeout=3).read().decode()[:300])
except Exception as e:
    print("ascent-api not reachable, trying localhost:", e)
    API = "http://localhost:8000"
    print(urllib.request.urlopen(API+"/api/health", timeout=3).read().decode()[:300])

In [ ]:
import urllib.request, json
def get(path):
    return json.loads(urllib.request.urlopen(API+path, timeout=5).read())
twins = get("/api/twins")
print(f"{len(twins)} twins")
for t in twins:
    print(f"- {t['id']}: {t['name']} [{t['mqtt_topic']}]")

In [ ]:
pts = get("/api/twins/esp32-demo/telemetry?limit=100")
print(f"{len(pts)} points, last:", pts[-1] if pts else None)
import matplotlib.pyplot as plt
if pts:
    xs = [p['time'][11:19] for p in pts]
    plt.figure(figsize=(10,3)); plt.plot(xs, [p.get('temperature') for p in pts]); plt.xticks(rotation=45)
    plt.title('esp32-demo temperature (live)'); plt.tight_layout(); plt.show()

## Push synthetic telemetry (same payload an ESP32 sends)

In [ ]:
import urllib.request, json, random
payload = json.dumps({"temperature": 23.5+random.random(), "humidity": 48+random.random()*5, "pressure": 1013.2, "co2": 445}).encode()
req = urllib.request.Request(API+"/api/twins/esp32-demo/telemetry", data=payload, headers={"Content-Type":"application/json"})
print(urllib.request.urlopen(req).read().decode())